In [ ]:
# ============================================================
#  신용평가 프로젝트 — M1 · M2 · M3 로지스틱 회귀 (통합본)
#  M1 = A(금융, 18)   M2 = B(비금융, 31)   M3 = A+B(49)
#  RQ1: 비금융이 금융에 예측력을 더하는가?  →  M3 vs M1 (DeLong)
# ============================================================
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from scipy import stats
import statsmodels.api as sm

try:
    from google.colab import files
    files.upload()   # model_A_financial_final.csv & model_B_nonfinancial_final.csv 둘 다 선택
except Exception:
    pass

PATH_A = "model_A_financial_final.csv"
PATH_B = "model_B_nonfinancial_final.csv"
N_BOOT = 1000        # 부트스트랩 반복 (느리면 500)

Saving model_B_nonfinancial_final.csv to model_B_nonfinancial_final.csv
Saving model_A_financial_final.csv to model_A_financial_final.csv


In [ ]:
a = pd.read_csv(PATH_A)
b = pd.read_csv(PATH_B)
if "AGE_BAND" in b.columns:
    b = b.drop(columns=["AGE_BAND"])              # EDA 전용(문자열) → 모델 제외
assert set(a["SK_ID_CURR"]) == set(b["SK_ID_CURR"]), "A/B SK_ID_CURR 불일치"

m = a.merge(b.drop(columns=["TARGET"]), on="SK_ID_CURR", how="inner")
y = m["TARGET"].astype(int).values
A_cols = [c for c in a.columns if c not in ("SK_ID_CURR", "TARGET")]
B_cols = [c for c in b.columns if c not in ("SK_ID_CURR", "TARGET")]
grp = {c: ("A" if c in A_cols else "B") for c in A_cols + B_cols}

print(f"rows={len(m):,}  결측={m.isna().sum().sum()}  연체율={y.mean():.4f}")
print(f"A(금융)={len(A_cols)}  B(비금융)={len(B_cols)}  M3(A+B)={len(A_cols+B_cols)}")
assert len(A_cols) == 18 and len(B_cols) == 31, "변수 개수 확인 필요"

rows=307,511  결측=0  연체율=0.0807
A(금융)=18  B(비금융)=31  M3(A+B)=49


In [ ]:
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5*(i+j-1)+1.0; i = j
    o = np.empty(N); o[J] = T; return o

def _fastdelong(preds, mpos):
    nt = preds.shape[1]; n = nt - mpos; k = preds.shape[0]
    pos, neg = preds[:, :mpos], preds[:, mpos:]
    tx = np.empty([k, mpos]); ty = np.empty([k, n]); tz = np.empty([k, nt])
    for r in range(k):
        tx[r] = _midrank(pos[r]); ty[r] = _midrank(neg[r]); tz[r] = _midrank(preds[r])
    aucs = tz[:, :mpos].sum(1)/mpos/n - (mpos+1.0)/2.0/n
    cov = np.cov((tz[:, :mpos]-tx)/n)/mpos + np.cov(1.0-(tz[:, mpos:]-ty)/mpos)/n
    return aucs, np.atleast_2d(cov)

def delong(y, s_full, s_reduced):
    """반환: ΔAUC, z, p(양측). DeLong AUC == sklearn roc_auc_score (사전 검증)."""
    y = np.asarray(y, float); order = (-y).argsort(kind="mergesort"); mpos = int(y.sum())
    preds = np.vstack((np.asarray(s_full)[order], np.asarray(s_reduced)[order]))
    aucs, cov = _fastdelong(preds, mpos)
    var = (np.array([[1., -1.]]) @ cov @ np.array([[1.], [-1.]])).item()
    d = aucs[0] - aucs[1]
    z, p = (0., 1.) if var <= 0 else (d/np.sqrt(var), 2*(1-stats.norm.cdf(abs(d/np.sqrt(var)))))
    return d, z, p

def boot_single(y, s, nb=None, seed=42):
    nb = nb or N_BOOT; rng = np.random.default_rng(seed)
    y = np.asarray(y); s = np.asarray(s); ip, ino = np.where(y==1)[0], np.where(y==0)[0]
    out = np.empty(nb)
    for i in range(nb):
        bi = np.concatenate([rng.choice(ip, ip.size, True), rng.choice(ino, ino.size, True)])
        out[i] = roc_auc_score(y[bi], s[bi])
    return tuple(np.percentile(out, [2.5, 97.5]))

def boot_diff(y, sf, sr, nb=None, seed=42):
    nb = nb or N_BOOT; rng = np.random.default_rng(seed)
    y = np.asarray(y); sf = np.asarray(sf); sr = np.asarray(sr)
    ip, ino = np.where(y==1)[0], np.where(y==0)[0]; out = np.empty(nb)
    for i in range(nb):
        bi = np.concatenate([rng.choice(ip, ip.size, True), rng.choice(ino, ino.size, True)])
        out[i] = roc_auc_score(y[bi], sf[bi]) - roc_auc_score(y[bi], sr[bi])
    return tuple(np.percentile(out, [2.5, 97.5]))

In [ ]:
def oof(cols):
    """스케일링을 폴드 내부에서만 fit → 데이터 누수 차단."""
    X = m[cols].values
    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    o = np.zeros(len(m))
    for tr, te in skf.split(X, y):
        pipe = Pipeline([("sc", StandardScaler()),
                         ("lr", LogisticRegression(C=1.0, max_iter=2000,
                                                   class_weight="balanced", solver="lbfgs"))])
        pipe.fit(X[tr], y[tr]); o[te] = pipe.predict_proba(X[te])[:, 1]
    return o

oof_M1, oof_M2, oof_M3 = oof(A_cols), oof(B_cols), oof(A_cols + B_cols)
AUC = {"M1": roc_auc_score(y, oof_M1), "M2": roc_auc_score(y, oof_M2), "M3": roc_auc_score(y, oof_M3)}
CI  = {k: boot_single(y, o) for k, o in [("M1", oof_M1), ("M2", oof_M2), ("M3", oof_M3)]}

model_tbl = pd.DataFrame([
    {"model": k, "X구성": x, "n_features": n, "OOF_AUC": round(AUC[k], 4),
     "CI_low": round(CI[k][0], 4), "CI_high": round(CI[k][1], 4)}
    for k, x, n in [("M1", "A_financial", len(A_cols)),
                    ("M2", "B_nonfinancial", len(B_cols)),
                    ("M3", "A+B", len(A_cols+B_cols))]])
print(model_tbl.to_string(index=False))

model            X구성  n_features  OOF_AUC  CI_low  CI_high
   M1    A_financial          18   0.6435  0.6400   0.6471
   M2 B_nonfinancial          31   0.6254  0.6218   0.6291
   M3            A+B          49   0.6797  0.6763   0.6830


In [ ]:
def compare(tag, sf, sr):
    d, z, p = delong(y, sf, sr); lo, hi = boot_diff(y, sf, sr)
    return {"비교": tag, "ΔAUC(%p)": round(d*100, 3), "DeLong_z": round(z, 2),
            "DeLong_p": f"{p:.2e}", "부트95%CI(%p)": f"[{lo*100:.2f}, {hi*100:.2f}]",
            "통계유의": p < 0.05, "실무유의(≥1%p)": d >= 0.01}

rq_df = pd.DataFrame([
    compare("RQ1: M3 vs M1", oof_M3, oof_M1),      # ★ 핵심
    compare("(참고) M3 vs M2", oof_M3, oof_M2),     # A의 기여
    compare("(참고) M2 vs M1", oof_M2, oof_M1),     # 비금융단독 vs 금융단독
])
print(rq_df.to_string(index=False))

           비교  ΔAUC(%p)  DeLong_z DeLong_p    부트95%CI(%p)  통계유의  실무유의(≥1%p)
RQ1: M3 vs M1     3.620     30.19 0.00e+00   [3.40, 3.87]  True        True
(참고) M3 vs M2     5.426     36.38 0.00e+00   [5.13, 5.72]  True        True
(참고) M2 vs M1    -1.806     -7.39 1.48e-13 [-2.28, -1.31]  True       False


In [ ]:
def coef_table(cols):
    """연속형만 표준화(더미는 0/1 → OR 해석). VIF = 상관행렬 역행렬 대각."""
    Xs = (m[cols]-m[cols].mean())/m[cols].std()
    vif = pd.Series(np.diag(np.linalg.pinv(np.corrcoef(Xs.values, rowvar=False))), index=cols)
    binary = [c for c in cols if m[c].nunique() <= 2]
    Xstd = m[cols].copy()
    for c in cols:
        if c not in binary and Xstd[c].std() > 0:
            Xstd[c] = (Xstd[c]-Xstd[c].mean())/Xstd[c].std()
    res = sm.Logit(y, sm.add_constant(Xstd)).fit(method="newton", maxiter=100, disp=0)
    star = lambda p: "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""
    t = pd.DataFrame({
        "variable": res.params.index, "group": [grp.get(c, "const") for c in res.params.index],
        "coef": res.params.values, "odds_ratio": np.exp(res.params.values),
        "p_value": res.pvalues.values, "sig": [star(p) for p in res.pvalues.values],
        "VIF": [vif.get(c, np.nan) for c in res.params.index]}).iloc[1:].reset_index(drop=True)
    t["_z"] = (t["coef"]/res.bse.values[1:]).abs()
    return t.sort_values("_z", ascending=False).drop(columns="_z").reset_index(drop=True)

coef_M1 = coef_table(A_cols)              # 금융 baseline 계수
coef_M3 = coef_table(A_cols + B_cols)     # 통합모델 계수 (A/B 그룹 태그)
print("M1 유의:", (coef_M1.sig != "").sum(), "/", len(coef_M1),
      " | M3 B군 유의:", (coef_M3[coef_M3.group=="B"].sig != "").sum(), "/", (coef_M3.group=="B").sum())
print("\nM3 B군 상위 동력:", ", ".join(coef_M3[coef_M3.group=="B"].head(5)["variable"]))

M1 유의: 16 / 18  | M3 B군 유의: 27 / 31

M3 B군 상위 동력: YEARS_EMPLOYED, AGE, OCCUPATION_TYPE_C_Core staff, OCCUPATION_TYPE_C_Accountants, OCCUPATION_TYPE_C_High skill tech staff


In [ ]:
with pd.ExcelWriter("model_M1M2M3_summary.xlsx", engine="openpyxl") as xw:
    model_tbl.to_excel(xw, sheet_name="01_모델비교", index=False)
    rq_df.to_excel(xw, sheet_name="02_RQ1검정", index=False)
    coef_M1.assign(p_value=coef_M1.p_value.map(lambda v: f"{v:.2e}")).to_excel(xw, sheet_name="03_M1계수", index=False)
    coef_M3.assign(p_value=coef_M3.p_value.map(lambda v: f"{v:.2e}")).to_excel(xw, sheet_name="04_M3계수", index=False)
    pd.DataFrame({"key": ["seed", "folds", "solver", "C", "class_weight", "n_boot"],
                  "value": [42, 5, "lbfgs", 1.0, "balanced", N_BOOT]}).to_excel(xw, sheet_name="05_설정", index=False)

pd.DataFrame({"SK_ID_CURR": m["SK_ID_CURR"], "oof_M1": oof_M1, "oof_M2": oof_M2,
              "oof_M3": oof_M3, "TARGET": y}).to_csv("oof_M1_M2_M3.csv", index=False)

r, r2 = rq_df.iloc[0], rq_df.iloc[2]
b_top = ", ".join(coef_M3[coef_M3.group=="B"].head(4)["variable"])
print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[신용평가 프로젝트] M1·M2·M3 결과 — 비금융 변수의 기여
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
■ 모델 (로지스틱 회귀 · 5-fold OOF AUC · EXT 제외)
   M1 금융(A,{len(A_cols)})          AUC {AUC['M1']:.4f}  [{CI['M1'][0]:.3f}-{CI['M1'][1]:.3f}]
   M2 비금융(B,{len(B_cols)})        AUC {AUC['M2']:.4f}  [{CI['M2'][0]:.3f}-{CI['M2'][1]:.3f}]
   M3 금융+비금융(A+B,{len(A_cols+B_cols)})  AUC {AUC['M3']:.4f}  [{CI['M3'][0]:.3f}-{CI['M3'][1]:.3f}]

■ RQ1: 비금융(B)이 금융(A)에 예측력을 더하는가?  →  YES
   M3 vs M1  ΔAUC = +{r['ΔAUC(%p)']}%p  (DeLong p<0.001, 95%CI {r['부트95%CI(%p)']}%p)
   → 통계적 유의 + 실무 임계(1~2%p) 초과

■ 핵심 해석
   · 비금융 단독(M2)은 금융 단독(M1)보다 낮음({r2['ΔAUC(%p)']}%p): 대체재 아님
   · A 위에 얹으면 +{r['ΔAUC(%p)']}%p 개선 → 상호보완적 신호
   · B {len(B_cols)}개 중 {(coef_M3[coef_M3.group=='B'].sig!='').sum()}개 유의. 주 동력: {b_top}
   · 소득유형·근무기관 더미 VIF 매우 높음(Working≈3316) → 계수 개별해석 자제, VIF 병기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━""")

try:
    from google.colab import files
    files.download("model_M1M2M3_summary.xlsx")
    files.download("oof_M1_M2_M3.csv")
except Exception:
    print("saved: model_M1M2M3_summary.xlsx, oof_M1_M2_M3.csv")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[신용평가 프로젝트] M1·M2·M3 결과 — 비금융 변수의 기여
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
■ 모델 (로지스틱 회귀 · 5-fold OOF AUC · EXT 제외)
   M1 금융(A,18)          AUC 0.6435  [0.640-0.647]
   M2 비금융(B,31)        AUC 0.6254  [0.622-0.629]
   M3 금융+비금융(A+B,49)  AUC 0.6797  [0.676-0.683]

■ RQ1: 비금융(B)이 금융(A)에 예측력을 더하는가?  →  YES
   M3 vs M1  ΔAUC = +3.62%p  (DeLong p<0.001, 95%CI [3.40, 3.87]%p)
   → 통계적 유의 + 실무 임계(1~2%p) 초과

■ 핵심 해석
   · 비금융 단독(M2)은 금융 단독(M1)보다 낮음(-1.806%p): 대체재 아님
   · A 위에 얹으면 +3.62%p 개선 → 상호보완적 신호
   · B 31개 중 27개 유의. 주 동력: YEARS_EMPLOYED, AGE, OCCUPATION_TYPE_C_Core staff, OCCUPATION_TYPE_C_Accountants
   · 소득유형·근무기관 더미 VIF 매우 높음(Working≈3316) → 계수 개별해석 자제, VIF 병기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>